# MLP V_θ — OpenWebText Scale-Up d=768 (Fock-PARFLM v2.1)

## Motivation

`colab_fock_depthcond_vtheta_openwebtext_d768.ipynb` scales the **structured**
V_θ — an explicit bank of Gaussian wells, depth-conditioned across the L Verlet
layers — to d=768.  Getting there required a stack of bounding knobs that exist
purely to contain the well parameterisation: `precision_max` tightened from
`2/d` to `1/d`, a new per-well `force_norm_max`, a total-force clamp, and the
global gradient clip halved.  Those are the fingerprints of the two failure
modes in `Training_Instabilities_in_Fock-PARFLM_with_structured_V_theta.md`:

1. **Precision runaway.** Each well carries a precision `a_k`.  When a well
   sharpens, the force near its centre spikes, and the spike propagates through
   all L integration steps.  The pressure grows with d, which is why d768
   needed bounds that d384 did not.
2. **Well collision.** Two well centres drifting together produce an
   ill-conditioned saddle — the dominant gradient-spike source at d ≥ 384.

This notebook runs the **control at the same scale**: identical Fock v2.1
architecture, d=768, with the default **MLP V_θ** (`ScalarPotentialMultiXi`),
an unstructured GELU MLP over `(ξ_1 … ξ_K, h)`.  Neither failure mode exists in
that parameterisation — a depth-3 MLP is Lipschitz-bounded by the product of
its layer spectral norms, and its curvature moves only as fast as the weights
do.  The question this run answers: **at d=768, is the well inductive bias
still worth the machinery needed to keep it stable?**

## The trade

| | MLP V_θ (this notebook) | Gaussian V_θ (depthcond d768) |
|---|---|---|
| Wells | implicit, learned | 40 explicit (5 heads × 8) |
| Force | `-∇_h MLP(ξ, h)` via autograd | analytical per-well gradient |
| Gradient spikes | rare, bounded | the reason for the d768 knob stack |
| `precision_max` / `force_norm_max` | no counterpart — nothing to bound | both required |
| Convergence | slower — the landscape must be discovered | faster — wells exist at init |
| V_θ params (d=768) | 47.2M at `v_hidden=3840` | 47.4M |

The arms are **parameter-matched**: 47.2M vs 47.4M V_θ params, so a PPL gap is
attributable to the inductive bias rather than the budget.  (Note that the
Gaussian bank grows roughly quadratically in d — 11.9M at d384, 47.4M at d768 —
so matching it here needs a much wider MLP than the `v_hidden=2048` used by the
d384 MLP arm.)

## What is held identical to the Gaussian d768 run

Same ξ channels (`5long`), V_φ (`structural_competitive`, 4 heads, `top_k=16`,
`d_type=32`, `d_angle=16`), WSD schedule, reverse-channel stabilisation
(E5c + per-layer gates), register repulsion, `TOTAL_STEPS`, and seed.

The d768 training knobs carry over too, including the ones introduced for
stability: `GRAD_CLIP=0.5`, `GRAD_CLIP_VPHI=0.2`, and
`force_clamp_max = 2/√768`.  Keeping them makes this a controlled comparison
rather than two runs that differ in several ways at once, and they cost nothing
diagnostically: the log records the **pre-clip** `V_theta` group gradient norm,
so the "are MLP forces smoother?" question is answered by the log regardless of
where the clip sits.  `force_norm_max` is the one d768 knob with no counterpart
here — it caps a per-well gradient, and there are no wells.

**Only V_θ changes.**

## Causal integrity

Both stages of the causal-leak audit run on a schedule (see
`Fock-PARFLM_Causal_Leak_Audit_Results.md`):

| Stage | What it measures | Cadence |
|---|---|---|
| **Architectural probe** | float64 forward on a tiny model with the same structural flags — perturbing the future must move past logits by exactly `0.0` | `CAUSAL_PROBE_INTERVAL` |
| **Trained-scale probe + honest PPL** | future-perturbation ΔNLL on real val windows at trained weights, plus leak-free last-position PPL vs the standard mid-window PPL | `TRAINED_LEAK_PROBE_INTERVAL` |

The architectural probe also runs once **before** step 1 and aborts the run on
failure, which is cheap insurance against burning H100 hours on a leaky model.
`prefix_causal_registers=True` is mandatory and asserted; checkpoints from
before that fix are **not** loadable.

## Hardware

Single **H100 80 GB** Colab instance.  Two things drive the footprint at d=768:
the prefix-causal register tensors, which are `(B, T, M, d)` per layer, and the
`create_graph=True` autograd over the V_θ MLP.  The micro-batch is auto-probed
(ceiling 4) and `GRAD_ACCUM=8` holds the effective batch fixed at whatever the
probe returns × 8.  Expect roughly twice the step time of the d384 MLP arm; the
periodic checkpoints plus nearest-checkpoint resume are what make a 100K-step
run survivable across Colab sessions.

## Prerequisites

- OpenWebText tokenised and cached on Google Drive.  Cell 3 reuses it from any
  earlier run directory (explicit list first, then a glob over sibling run
  dirs), so this should never re-tokenise.

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────
import math as _math

# ── V_theta architecture ────────────────────────────────────────────
# This notebook is the MLP arm: V_theta stays the model's built-in
# ScalarPotentialMultiXi (a GELU MLP over concat(xi_1..xi_K, h)), i.e. it is
# NOT swapped for a Gaussian / SARF / SQ3 well bank.  The variant is kept as
# an explicit knob so the cell reads the same as the structured notebooks and
# so a Gaussian control can be run from this file if ever needed.
V_THETA_VARIANT = 'mlp'         # 'mlp' for this experiment

# MLP width / depth.  Param count is
#     (K+1)*d*H + L_v*H + (L_v-1)*H^2 + 1
# With d=768, K=5 (xi5long), H=3840, L_v=3 that is 47.2M — matched to the 47.4M
# of the depth-conditioned Gaussian bank at d768, so the two arms differ in
# inductive bias, not in parameter budget.  (The Gaussian bank grows ~d^2:
# 11.9M at d384 -> 47.4M at d768, hence the much wider MLP than the d384 arm's
# 2048.)  Drop to 2048 (17.8M) for a cheaper, non-matched run.
V_THETA_HIDDEN = 3840           # v_hidden  (2048 -> 17.8M, 3840 -> 47.2M at d=768)
V_THETA_DEPTH  = 3              # v_depth   (number of hidden GELU blocks)

# ── V_theta force bounding ──────────────────────────────────────────
# FORCE_CLAMP_MAX is a per-dimension clamp on the TOTAL conservative force
# (V_theta + V_phi) after autograd, applied in model_parf_multixi._layer_step.
# It is NOT Gaussian-specific, and the d768 Gaussian baseline runs with it, so
# it is ON here at the same value: dropping it would make this arm differ from
# the baseline in two places instead of one.
#
# The Gaussian run's other bound, force_norm_max, has no counterpart here — it
# caps a PER-WELL gradient inside analytical_grad, and the MLP has no wells and
# no analytical_grad.  Nor is there a precision parameter to bound: the MLP's
# Lipschitz constant is the product of its layer spectral norms and moves only
# as fast as the weights do.  That absence is the hypothesis under test.
#
# LN_BEFORE_VTHETA puts a LayerNorm on h before V_theta, bounding the MLP's
# input range at ~zero cost.  Off by default (the Gaussian baseline has no
# equivalent); it is the cheapest first response if the '[spike]' lines start
# naming the 'V_theta' group.  Changing either knob mid-run makes the resumed
# segment non-comparable with what came before.
LN_BEFORE_VTHETA = False              # LayerNorm(h) before V_theta
FORCE_CLAMP_MAX  = 2.0 / _math.sqrt(768)   # ~= 0.0722; matches Gaussian d768

# ── Xi channel override ──────────────────────────────────────────────
XI_OVERRIDE     = '5long'       # '5long' (default) | 5 | 6 | '4long' | None

_XI_PRESETS_CFG = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
if XI_OVERRIDE is None:
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
elif XI_OVERRIDE in _XI_PRESETS_CFG:
    XI_ALPHA_INITS = _XI_PRESETS_CFG[XI_OVERRIDE]
else:
    raise ValueError(f'Unsupported XI_OVERRIDE={XI_OVERRIDE!r}; use None, 5, 6, "5long", or "4long"')
XI_CHANNELS = len(XI_ALPHA_INITS)

# ── PARF V_phi knobs (identical to the Gaussian d384 run) ────────
V_PHI_KIND      = 'structural_competitive'  # 'structural_competitive' | 'structural' | 'mlp'
V_PHI_MLP_HIDDEN = 128
TOP_K           = 16            # sparse pairs per query
V_PHI_N_HEADS   = 4
V_PHI_D_TYPE    = 32            # type-vector dimension d_l
V_PHI_D_ANGLE   = 16            # value-angle dimension K

# ── Fock reverse-channel stabilisation (E5c) ─────────────────────
# Unchanged from the Gaussian d384 run so the reverse channel is not a
# confound in the V_theta comparison.  See §10.12 of
# Improving_the_Fock_Mechanism_to_match_Attention.md for each knob.
REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE       = True
REVERSE_CHANNEL_PRE_LN       = True
REVERSE_CHANNEL_SOFT_NORM    = True
REVERSE_CHANNEL_WARMUP_STEPS = 4000
REVERSE_CHANNEL_PER_LAYER    = True
# One-time transitional reset when resuming a PRE-stable checkpoint into the
# stabilised reverse channel.  Leave False for a fresh run.
REVERSE_CHANNEL_RESET_SCALE  = False

# ── Register repulsion (B4) — anti-collapse restoring force ──────
REGISTER_REPULSION       = True
REGISTER_REPULSION_COEFF = 0.05
REGISTER_REPULSION_KIND  = 'gram'   # 'gram' (sq off-diag cosine) | 'coulomb'

# ── Output read-out head ─────────────────────────────────────────
USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False         # untied W_out

# ── Optimizer ─────────────────────────────────────────────────────
OPTIMIZER = 'adamw'             # 'adamw' | 'lamb' | 'lion'
GRAD_CENTRALIZATION = False

# ── LR schedule ──────────────────────────────────────────────────
LR_SCHEDULE     = 'wsd'         # 'cosine' | 'wsd'
WSD_WARMUP_FRAC = 0.05          # fraction of TOTAL_STEPS for linear warmup
WSD_STABLE_FRAC = 0.60          # fraction at peak LR (plateau phase)
WSD_LR_FLOOR    = None          # resolved to LR * 0.05 in the training cell

# ── Batch / accumulation ─────────────────────────────────────────
# Two things shrink the micro-batch at d=768: the prefix-causal register
# tensors are (B, T, M, d) per layer, and autograd.grad(create_graph=True)
# keeps the whole V_theta MLP graph alive.  GRAD_ACCUM is raised to 8 (vs 4 at
# d384, 4 in the Gaussian d768 run) so the effective batch stays in range even
# when the probe settles on a micro-batch of 2-4.
GRAD_ACCUM      = 8

# ── Variant tag ──────────────────────────────────────────────────
_variant_parts = [f'mlpvt_h{V_THETA_HIDDEN}d{V_THETA_DEPTH}']
if XI_OVERRIDE is not None:
    _variant_parts.append(f'xi{XI_OVERRIDE}')
if TOP_K != 8:
    _variant_parts.append(f'topk{TOP_K}')
if V_PHI_D_TYPE != 16 or V_PHI_D_ANGLE != 8:
    _variant_parts.append(f'dt{V_PHI_D_TYPE}da{V_PHI_D_ANGLE}')
if V_PHI_N_HEADS != 1:
    _variant_parts.append(f'mh{V_PHI_N_HEADS}')
if LN_BEFORE_VTHETA:
    _variant_parts.append('lnvt')
if FORCE_CLAMP_MAX is not None:
    _variant_parts.append(f'fc{FORCE_CLAMP_MAX:.3g}')
if USE_OUTPUT_BIAS:
    _variant_parts.append('ob')
if not TIE_EMBEDDINGS:
    _variant_parts.append('untied')
if OPTIMIZER != 'adamw':
    _variant_parts.append(OPTIMIZER)
if GRAD_CENTRALIZATION:
    _variant_parts.append('gc')
if LR_SCHEDULE != 'cosine':
    _variant_parts.append(LR_SCHEDULE)
if not REVERSE_CHANNEL:
    _variant_parts.append('e5a')
elif REVERSE_CHANNEL_STABLE:
    _variant_parts.append('e5c')
if REVERSE_CHANNEL and REVERSE_CHANNEL_PER_LAYER:
    _variant_parts.append('plgate')
if REGISTER_REPULSION:
    _variant_parts.append(f'rep{REGISTER_REPULSION_COEFF:g}')
_variant_tag = '_'.join(_variant_parts)

# ── V_theta param preview (d resolved later from ARCH_TIERS) ─────
def _mlp_vtheta_params(d, K=XI_CHANNELS, H=V_THETA_HIDDEN, Lv=V_THETA_DEPTH):
    """Param count of ScalarPotentialMultiXi: (K+1)d->H, (Lv-1) x H->H, H->1."""
    return (K + 1) * d * H + H + (Lv - 1) * (H * H + H) + H + 1

print(f'Config: V_theta={V_THETA_VARIANT} (unstructured MLP)')
print(f'  v_hidden={V_THETA_HIDDEN}  v_depth={V_THETA_DEPTH}')
for _d in (768, 384):
    print(f'    d={_d}: V_theta params ~= {_mlp_vtheta_params(_d):,}')
print(f'  force bounding: ln_before_vtheta={LN_BEFORE_VTHETA}  '
      f'force_clamp_max={FORCE_CLAMP_MAX}')
if XI_OVERRIDE is not None:
    print(f'  XI_OVERRIDE={XI_OVERRIDE} -> {XI_CHANNELS}ch, '
          f'horizons ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tok')
print(f'  V_phi={V_PHI_KIND}  heads={V_PHI_N_HEADS}  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE} (eff {V_PHI_N_HEADS*V_PHI_D_TYPE})  '
      f'd_angle={V_PHI_D_ANGLE} (eff {V_PHI_N_HEADS*V_PHI_D_ANGLE})')
print(f'  read-out: output_bias={USE_OUTPUT_BIAS}, '
      + ('tied E^T' if TIE_EMBEDDINGS else 'UNTIED W_out'))
print(f'  optimizer={OPTIMIZER}  grad_centralization={GRAD_CENTRALIZATION}')
print(f'  LR schedule={LR_SCHEDULE}  grad_accum={GRAD_ACCUM}')
print(f'  [variant] tag={_variant_tag}')

In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_mlp_vtheta_owt_d768'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    # The tokenised OWT cache lives on Drive and is symlinked into the repo so
    # data_module resolves it exactly as it would locally.  Surviving a runtime
    # restart is the whole point: re-tokenising 1B tokens costs ~1 h.
    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'mlp_vtheta_d768' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_mlpvt_owt_d768' + (f'_{_variant_tag}' if _variant_tag else '')
TOTAL_STEPS   = 100_000
CKPT_INTERVAL = 7_500
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# ── Cell 2: Checkpoint resolution + resume detection ─────────────
# Resumes from the NEAREST usable checkpoint: the highest periodic step file,
# unless a '_best' checkpoint is more recent (which happens when the session
# died between a best-eval and the next periodic save).
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

resume_step = 0
resume_ckpt = None

for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_ppl = float('nan')
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) — resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

if resume_ckpt is not None:
    print(f'\nResuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found — training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# ── Cell 3: Data loading (reuse cached OpenWebText) ──────────────
from data_module import get_batch

MAX_TRAIN_TOKENS = 1_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000
VOCAB_SIZE       = 50257

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

# Every earlier OWT run dir is a candidate donor for the tokenised cache: the
# token stream is identical across experiments, only the model differs.  First
# hit wins, so the Gaussian d384/d768 dirs (most likely to exist) come first.
for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_depthcond_vtheta_owt',
    'semsimula_fock_depthcond_vtheta_owt_d768',
    'semsimula_fock_mlp_vtheta_owt_d384_mlpvt_h2048d3_xi5long_topk16_dt32da16_mh4_ob_untied_wsd_e5c_plgate_rep0.05',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists() and alt_val.exists():
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

# Fallback: any sibling run directory will do, since the token stream is
# identical across experiments and run dirs carry variant tags that cannot be
# enumerated ahead of time.
if not train_cache.exists():
    _search_root = Path('/content/drive/MyDrive') if IN_COLAB else Path.home()
    for _cand in sorted(_search_root.glob(
            f'semsimula_*/data/openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy')):
        _cand_val = _cand.parent / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
        if _cand_val.exists():
            print(f'Reusing data cache from {_cand.parent}  (glob fallback)')
            shutil.copy2(str(_cand), str(train_cache))
            shutil.copy2(str(_cand_val), str(val_cache))
            break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# ── Cell 4: Model config + MLP V_theta ───────────────────────────
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
from model_multixi import ScalarPotentialMultiXi
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

_XI_PRESETS = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
if XI_OVERRIDE is None:
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
elif XI_OVERRIDE in _XI_PRESETS:
    XI_ALPHA_INITS = _XI_PRESETS[XI_OVERRIDE]
else:
    raise ValueError(f'Unsupported XI_OVERRIDE={XI_OVERRIDE!r}; use None, 5, 6, "5long", or "4long"')
XI_CHANNELS = len(XI_ALPHA_INITS)
print(f'Xi: {XI_CHANNELS} channels, alphas={XI_ALPHA_INITS}')
print(f'  Horizons: ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tokens')

LAMBDA_V       = 1e-2
BLOCK_SIZE     = 512

LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

# d=768 is the target tier; the fallbacks trade depth, then registers, then
# width, so an unexpectedly small card still produces a runnable model rather
# than an OOM at cell-run time.  Mirrors the Gaussian d768 ladder.
ARCH_TIERS = [
    (768, 24, 32),
    (768, 16, 32),
    (768, 16, 16),
    (384, 16, 32),
]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L,
        # V_theta = ScalarPotentialMultiXi(d, hidden=v_hidden, depth=v_depth,
        #                                  K=xi_channels)  -- the MLP arm.
        v_hidden=V_THETA_HIDDEN, v_depth=V_THETA_DEPTH,
        dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=0.30,
        causal_force=True,
        ln_after_step=True,
        # MLP V_theta force bounding (both default OFF; see Cell 0).
        ln_before_vtheta=LN_BEFORE_VTHETA,
        force_clamp_max=FORCE_CLAMP_MAX,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_eps=0.1,
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        reverse_channel_per_layer=REVERSE_CHANNEL_PER_LAYER,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
        register_repulsion=REGISTER_REPULSION,
        register_repulsion_coeff=REGISTER_REPULSION_COEFF,
        register_repulsion_kind=REGISTER_REPULSION_KIND,
        # Prefix-causal register lifecycle (causal-leak fix; see
        # Fock-PARFLM_Causal_Leak_Audit_Results.md).  Must be True for any
        # trustworthy run.  Pre-fix checkpoints are NOT loadable.
        prefix_causal_registers=True,
    )


# ── Try architecture tiers ─────────────────────────────────────────
model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
        # No V_theta swap: this is the MLP arm.  Assert rather than trust, so a
        # future edit that reintroduces a structured swap fails loudly here
        # instead of silently invalidating the comparison.
        assert isinstance(mdl.V_theta, ScalarPotentialMultiXi), (
            f'expected the built-in MLP V_theta, got {type(mdl.V_theta).__name__}')
        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta MLP {n_v_theta:,}, h={V_THETA_HIDDEN} x {V_THETA_DEPTH})')
        if DEVICE == 'cuda':
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = mdl(_x, _y)
            _loss.backward()
            mdl.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} — trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

assert model_cfg.prefix_causal_registers, 'prefix-causal register fix must be ON'

if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# ── Auto batch size (GRAD_ACCUM is fixed from Cell 0) ──────────────
# Ceiling is 4: at d=768 the (B, T, M, d) register tensors and the retained
# V_theta MLP graph both scale with the micro-batch, and the tier ladder may
# have landed on L=24.  GRAD_ACCUM=8 recovers the effective batch.
BATCH_SIZE = 1
if DEVICE == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    _probe_sizes = [4, 3, 2, 1] if _vram_gb >= 70 else [2, 1]
    for bs in _probe_sizes:
        try:
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, bs, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            model.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            BATCH_SIZE = bs
            print(f'Auto batch: {bs} x accum={GRAD_ACCUM} (eff={bs*GRAD_ACCUM})')
            break
        except RuntimeError:
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
# Eval runs the same force computation but without create_graph, so it fits at
# the training micro-batch.  Halve this first if eval OOMs.
EVAL_BATCH = BATCH_SIZE

n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())

print(f'\nModel: FockMultiXiPARFLM v2.1 + MLP V_theta  (d768 arm)')
print(f'  params: {n_params:,}  (V_theta MLP: {n_v_theta:,} = '
      f'{100*n_v_theta/n_params:.1f}% of total)')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta: in_dim={(XI_CHANNELS+1)*model_cfg.d}  hidden={V_THETA_HIDDEN}  '
      f'depth={V_THETA_DEPTH}  lambda_V={LAMBDA_V}')
print(f'  force bounding: ln_before_vtheta={model_cfg.ln_before_vtheta}  '
      f'force_clamp_max={model_cfg.force_clamp_max}')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})  '
      f'eval_batch={EVAL_BATCH}')
print(f'  prefix_causal_registers={model_cfg.prefix_causal_registers}')
if DEVICE == 'cuda':
    print(f'  peak VRAM after probes: '
          f'{torch.cuda.max_memory_allocated()/1e9:.1f} / {_vram_gb:.0f} GB')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────

LR            = 3e-4      # peak LR (WSD keeps it here through the plateau)
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = int(WSD_WARMUP_FRAC * TOTAL_STEPS) if LR_SCHEDULE == 'wsd' else 4000
# Both tightened to match the Gaussian d768 run.  They are not V_theta
# specific, so keeping the baseline's values leaves V_theta as the only
# difference between the arms.  The pre-clip V_theta group norm is logged every
# LOG_INTERVAL, so a tighter clip does not hide the smoothness signal — if that
# norm stays well under 0.5, relaxing back to 1.0 is a justified follow-up.
GRAD_CLIP     = 0.5
GRAD_CLIP_VPHI = 0.2

# ── Per-group (per-layer) gradient clipping ──────────────────────
# Each top-level module's gradients are clipped to their OWN max-norm instead
# of a single global rescale, so one exploding component cannot either dominate
# the global norm (zeroing every useful gradient) or slip under it.  Substring
# matches in GRAD_CLIP_OVERRIDES take priority over the per-module default.
#
# Note for this arm: V_theta is NOT overridden, so the MLP falls in its own
# 'V_theta' group at the default GRAD_CLIP.  That group is the one to watch in
# the '[spike]' lines — it is the direct read-out of whether the unstructured
# potential is behaving as smoothly as the Gaussian analysis predicts.
PER_GROUP_CLIP = True
GRAD_CLIP_OVERRIDES = {
    'V_phi': GRAD_CLIP_VPHI,   # pairwise potential: keep the tight 0.3 clip
    'creation_gate': 0.3,      # Fock QKV creation gate (W_Q / W_K / log_tau)
    'destruction_gate': 0.3,   # Fock destruction gates
    # The reverse-channel GATE is a single scalar whose gradient sums over
    # B*T*L positions, so it is naturally O(1e3+) even when the (RMS-normed,
    # warmup-gated) force it controls is ~0.  Give it its own group (matched
    # BEFORE 'reverse_ch') and exclude it from the watchdog total below, so
    # this benign scalar can't trigger spurious reloads.
    'reverse_channel_scale': 0.1,
    'reverse_ch': 0.1,         # Fock reverse-channel projections + LN + logit_scale
    'register': 0.3,           # register_embed (Fock register bank)
}
# Groups clipped and logged but EXCLUDED from the global pre-clip norm that
# drives the spike debugger and watchdog.  Both reverse-channel groups are
# per-group clipped to <=0.1, so their UPDATE is bounded regardless of pre-clip
# norm; the large pre-clip value is the 1/||Q|| Jacobian of the output norm
# acting on a small natural force, not a stability signal.
WATCHDOG_EXCLUDE_GROUPS = {'override:reverse_channel_scale', 'override:reverse_ch'}

# ── Per-step gradient-spike debugger ─────────────────────────────
GRAD_SPIKE_DEBUG     = True
GRAD_SPIKE_THRESHOLD = 100.0   # pre-clip total norm that counts as a spike
GRAD_SPIKE_COOLDOWN  = 0       # min steps between spike prints (0 = every spike)

EVAL_INTERVAL = 500    # PPL update every ~500 steps
EVAL_ITERS    = 40
LOG_INTERVAL  = 50     # training loss line every ~50 steps

# ── Two-stage causal-leak audit (see the header markdown) ────────
# Stage 1 is architectural: a tiny float64 model with the same structural
# flags, where any nonzero past-logit response to a future perturbation is a
# bug.  Stage 2 is empirical: the live trained weights on real val windows,
# where a nonzero response is a measurement of how much PPL the leak is
# worth.  Stage 1 catching nothing while stage 2 fires means the leak is in
# the trained dynamics, not the wiring.
CAUSAL_PROBE_INTERVAL       = 20_000  # architectural probe (0 = off)
TRAINED_LEAK_PROBE_INTERVAL = 20_000  # trained-scale probe + honest PPL (0 = off)
TRAINED_LEAK_PROBE_K        = 256     # honest-PPL targets (256 ~2 min; 1024 ~8 min)
TRAINED_LEAK_PROBE_PAIRS    = 2       # future-perturbation window pairs
SEED          = 0

# Resolve WSD_LR_FLOOR now that LR is final.
if WSD_LR_FLOOR is None:
    WSD_LR_FLOOR = LR * 0.05

GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 50.0
GRAD_NORM_EMA_PATIENCE = 200

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    """Unified LR schedule supporting cosine and WSD."""
    if LR_SCHEDULE == 'wsd':
        warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
        stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
        if step < warmup_end:
            return LR * (step + 1) / max(warmup_end, 1)
        elif step < stable_end:
            return LR
        else:
            decay_steps = TOTAL_STEPS - stable_end
            progress = (step - stable_end) / max(decay_steps, 1)
            cos_decay = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
            return WSD_LR_FLOOR + (LR - WSD_LR_FLOOR) * cos_decay
    else:
        if step < WARMUP_STEPS:
            return LR * (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
        return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        # The MLP potential is unbounded, so the plain square penalty is the
        # right regulariser here (the log1p form exists for SQ3, whose values
        # can blow up quadratically in h).
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, EVAL_BATCH, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        # The force needs autograd even in eval, so grad is re-enabled inside
        # the no_grad wrapper; create_graph is False because model.eval().
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def run_causal_probe(step_num):
    """Stage 1: architectural causal-integrity probe (CPU, float64).

    Builds a tiny model carrying the same structural flags as the training
    config -- including the MLP V_theta of this arm, so the probe exercises the
    same force path -- opens the reverse channel fully, and checks that
    perturbing future tokens moves earlier-position logits by exactly zero.
    A tiny model suffices because the property under test is structural: it
    holds for all weights or for none.

    Returns (passed: bool, max_delta: float).
    """
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        # MLP V_theta, scaled down but structurally identical to the run.
        v_hidden=64, v_depth=V_THETA_DEPTH, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(_logfreq_probe),
        logfreq_init_alpha=0.1, init_gamma=1.0, fixed_gamma=0.30,
        causal_force=True, ln_after_step=True,
        ln_before_vtheta=LN_BEFORE_VTHETA, force_clamp_max=FORCE_CLAMP_MAX,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=8, v_phi_d_angle=4, v_phi_eps=0.1,
        v_phi_phi_hidden=16, v_phi_theta_hidden=16, v_phi_mlp_hidden=16,
        top_k=8, v_phi_n_heads=2,
        use_output_bias=True, tie_embeddings=False,
        score_head_hidden=8,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True, reverse_channel_stable=True,
        reverse_channel_pre_ln=True, reverse_channel_soft_norm=True,
        reverse_channel_warmup_steps=4000, reverse_channel_per_layer=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, register_repulsion=False,
        prefix_causal_registers=True,
    )
    torch.manual_seed(1234)
    _probe_model = FockMultiXiPARFLM(_probe_cfg)   # keeps the MLP V_theta
    _probe_model.double().eval()

    # Worst case for the probe: gate wide open and warmup complete, so the
    # reverse channel carries its full force.
    with torch.no_grad():
        _probe_model.reverse_channel_scale.fill_(1.0)
        _probe_model.reverse_warmup_step.fill_(4000)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    with torch.enable_grad():
        _la = _probe_model(_x1)[0].detach()
        _lb = _probe_model(_x2)[0].detach()
    _max_delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())

    # Train-mode check (STE gumbel path).  Seeds are pinned so the two forwards
    # draw the SAME gumbel noise -- otherwise the noise itself, not a leak,
    # would show up as a nonzero delta.
    _probe_model.train()
    torch.manual_seed(99)
    with torch.enable_grad():
        _lta = _probe_model(_x1)[0].detach()
    torch.manual_seed(99)
    with torch.enable_grad():
        _ltb = _probe_model(_x2)[0].detach()
    _probe_model.eval()
    _max_delta_train = float((_lta[:, :_t_p] - _ltb[:, :_t_p]).abs().max().item())

    _max_delta = max(_max_delta, _max_delta_train)
    _passed = (_max_delta == 0.0)

    del _probe_model, _la, _lb, _lta, _ltb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  '
          f'(eval={_max_delta:.3e}, train={_max_delta_train:.3e})  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
        print('[causal probe] The prefix_causal_registers fix may not be working correctly.')
    return _passed, _max_delta


def run_trained_leak_probe(step_num):
    """Stage 2: trained-scale leak probe + honest PPL on the live model.

    Part 1 perturbs the future half of real validation windows and measures the
    effect on past logits / past-target NLL at the trained gate values.
    Part 2 scores the same targets mid-window (what the training loss and the
    in-loop eval both do) against last-position (leak-free by construction);
    the gap is the PPL the leak is worth.

    Both helpers leave the model in eval(), so train() is restored before
    returning.
    """
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} — running on live model')
    print(f'{"="*64}')

    # float64 is off here: this runs on the GPU mid-training, and the effect
    # sizes that matter are far above float32 noise.
    probe_res = probe_trained_leak(
        model, val_ids, device=DEVICE, context=BLOCK_SIZE,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)

    honest_res = honest_ppl_test(
        model, val_ids, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK_SIZE, batch=EVAL_BATCH, device=DEVICE)

    model.train()

    result = {
        'step': step_num,
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'grad_clip_vphi': GRAD_CLIP_VPHI,
            'optimizer': OPTIMIZER, 'grad_centralization': GRAD_CENTRALIZATION,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
            'lr_schedule': LR_SCHEDULE,
            'v_theta_hidden': V_THETA_HIDDEN,
            'v_theta_depth': V_THETA_DEPTH,
            'ln_before_vtheta': LN_BEFORE_VTHETA,
            'force_clamp_max': FORCE_CLAMP_MAX,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': 'fock_parf_multixi_v2.1_mlp_vtheta',
        'corpus': 'openwebtext',
        'phase': 6,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    # Drive occasionally drops the FUSE mount mid-run (errno 107); one
    # remount-and-retry turns a lost run into a lost minute.
    for _attempt in range(2):
        try:
            torch.save(ckpt, path)
            break
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error saving checkpoint; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}')
                    print(f'[WARN] Checkpoint NOT saved: {path}')
                    return None
            else:
                print(f'[WARN] Checkpoint save failed: {_e}')
                return None
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# ── Optimizer ──
_trainable = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER == 'adamw':
    optim = torch.optim.AdamW(_trainable, lr=LR,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lamb':
    try:
        import torch_optimizer
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_optimizer'])
        import torch_optimizer
    optim = torch_optimizer.Lamb(_trainable, lr=LR,
                                 weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lion':
    try:
        from lion_pytorch import Lion
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lion-pytorch'])
        from lion_pytorch import Lion
    optim = Lion(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
else:
    raise ValueError(f'Unknown OPTIMIZER={OPTIMIZER!r}; choose adamw / lamb / lion')
print(f'Optimizer: {type(optim).__name__}')

# ── Resume ──
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    _prev_tcfg = ckpt_data.get('train_cfg', {})
    _prev_h = _prev_tcfg.get('v_theta_hidden')
    _prev_d = _prev_tcfg.get('v_theta_depth')
    if _prev_h is not None and (_prev_h, _prev_d) != (V_THETA_HIDDEN, V_THETA_DEPTH):
        raise RuntimeError(
            f'Checkpoint was trained with v_hidden={_prev_h} v_depth={_prev_d}, '
            f'but this notebook is configured for v_hidden={V_THETA_HIDDEN} '
            f'v_depth={V_THETA_DEPTH}. Shapes will not match — change the config '
            f'back or start a fresh run directory.')
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if (REVERSE_CHANNEL and REVERSE_CHANNEL_STABLE and REVERSE_CHANNEL_RESET_SCALE
            and getattr(model, 'reverse_channel_scale', None) is not None):
        with torch.no_grad():
            model.reverse_channel_scale.zero_()
            if hasattr(model, 'reverse_warmup_step'):
                model.reverse_warmup_step.zero_()
        print('  [E5c] reverse_channel_scale re-zeroed + warmup reset for clean stable start')
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Training state ──
log_path = RESULTS_DIR / 'training_log.jsonl'
_log_fh = [log_path.open('a')]

def _log_write(record_str):
    """Write a JSONL record to the Drive log, remounting on transport error."""
    for _attempt in range(2):
        try:
            _log_fh[0].write(record_str)
            _log_fh[0].flush()
            return
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error on log write; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                    try:
                        _log_fh[0].close()
                    except Exception:
                        pass
                    _log_fh[0] = log_path.open('a')
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}; log record lost, training continues.')
                    return
            else:
                print(f'[WARN] Log write failed (attempt {_attempt+1}): {_e}; training continues.')
                return

t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        print(f'No canonical _best.pt; using {_best_ckpt_path.name}')
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical
        print(f'  Copied to canonical: {_canonical.name}')

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0

def _reload_best():
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s

steps_this_session = 0

# ── Schedule summary ──
if LR_SCHEDULE == 'wsd':
    _warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
    _stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
    _sched_str = (f'WSD: warmup 0->{_warmup_end:,}, stable {_warmup_end:,}->{_stable_end:,}, '
                  f'decay {_stable_end:,}->{TOTAL_STEPS:,}, floor={WSD_LR_FLOOR:.2e}')
else:
    _sched_str = f'cosine: warmup {WARMUP_STEPS:,} steps'

print(f'\n{"="*60}')
print(f'MLP V_theta (unstructured, d768): steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  grad_clip={GRAD_CLIP}  grad_clip_vphi={GRAD_CLIP_VPHI}')
print(f'  schedule: {_sched_str}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta: MLP hidden={V_THETA_HIDDEN} depth={V_THETA_DEPTH}  '
      f'params={n_v_theta:,}  (model total {n_params:,})')
print(f'  force bounding: ln_before_vtheta={LN_BEFORE_VTHETA}  '
      f'force_clamp_max={FORCE_CLAMP_MAX}')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE}')
if PER_GROUP_CLIP:
    print(f'  per-group clip: default={GRAD_CLIP}  overrides={GRAD_CLIP_OVERRIDES}')
if REVERSE_CHANNEL:
    _rev_mode = ('stable (QK-norm + '
                 + ('soft-norm' if REVERSE_CHANNEL_SOFT_NORM else 'RMS-norm')
                 + (' + pre-LN' if REVERSE_CHANNEL_PRE_LN else '') + ')'
                 ) if REVERSE_CHANNEL_STABLE else 'vanilla'
    print(f'  reverse channel: {_rev_mode}  warmup={REVERSE_CHANNEL_WARMUP_STEPS} forwards')
else:
    print('  reverse channel: OFF (E5a ablation)')
print(f'  causal probe every {CAUSAL_PROBE_INTERVAL:,} steps; '
      f'trained-leak probe every {TRAINED_LEAK_PROBE_INTERVAL:,} steps')
print(f'{"="*60}\n')


def _assign_clip_group(pname):
    """Map a parameter name to (group_key, max_norm).

    Override substrings win (so every V_phi / Fock tensor is clipped as one
    tight group); otherwise the parameter is grouped by its top-level module
    and clipped to the default GRAD_CLIP.
    """
    low = pname.lower()
    for sub, thr in GRAD_CLIP_OVERRIDES.items():
        if sub.lower() in low:
            return f'override:{sub}', thr
    return pname.split('.', 1)[0], GRAD_CLIP


def per_group_grad_norms(model):
    """Non-mutating snapshot of pre-clip per-group gradient norms."""
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        key, _ = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
    out = {}
    for key, ps in groups.items():
        sq = 0.0
        for p in ps:
            sq += float(p.grad.detach().norm()) ** 2
        out[key] = sq ** 0.5
    return out


def clip_grads_per_group(model):
    """Clip gradients independently per module group.

    Returns (total_norm_tensor, per_group_norms_dict).  total_norm is the
    global pre-clip norm (sqrt of summed per-group squared norms) over the
    non-excluded groups, so the watchdog threshold stays comparable across
    experiments.
    """
    groups, thr = {}, {}
    _dev = None
    for n, p in model.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        if _dev is None:
            _dev = p.grad.device
        key, mx = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
        thr[key] = mx
    total_sq = torch.zeros((), device=_dev) if _dev is not None else torch.zeros(())
    per_group = {}
    for key, ps in groups.items():
        gn = nn.utils.clip_grad_norm_(ps, thr[key])
        per_group[key] = float(gn)
        if key not in WATCHDOG_EXCLUDE_GROUPS:
            total_sq = total_sq + gn.detach() ** 2
    return total_sq.sqrt(), per_group


# ── Pre-flight: confirm the clip groups resolved as intended ──
print('exclude set        :', WATCHDOG_EXCLUDE_GROUPS)
_vt_group = _assign_clip_group('V_theta.net.0.weight')
print(f'V_theta clip group : {_vt_group[0]}  max_norm={_vt_group[1]}')
if getattr(model, 'reverse_ch', None) is not None:
    print('stable / pre_ln    :', model.reverse_ch.stable, model.reverse_ch.pre_ln)
    _gate = model.reverse_channel_scale.detach()
    print('gate value         :', float(_gate) if _gate.numel() == 1
          else f'per-layer[{_gate.numel()}] mean={float(_gate.mean()):.4f}')
else:
    print('reverse channel    : OFF (no reverse_ch module)')

# ── Run the causal probes once before training ──
# A fresh run should be provably clean at step 0; a resumed run should be
# clean at the step it restarts from.  Either way, finding a leak now is far
# cheaper than finding it 20k steps in.
if CAUSAL_PROBE_INTERVAL > 0:
    _cp0_passed, _cp0_delta = run_causal_probe(resume_step)
    _log_write(json.dumps({
        'step': resume_step, 'phase': 'pre_train',
        'causal_probe_passed': _cp0_passed,
        'causal_probe_max_delta': _cp0_delta,
    }) + '\n')
    if not _cp0_passed:
        raise RuntimeError(
            'Architectural causal probe FAILED before training started. '
            'Refusing to burn H100 hours on a leaky model — fix the wiring first.')

_last_pg_norms = {}
_last_spike_step = -10**9
for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    accum_rep = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg = forward_with_vreg(x, y, LAMBDA_V)
        # Register repulsion (B4): drain the per-layer penalty from THIS
        # forward and fold it in before backward (kept out of the forward so it
        # never enters the eval PPL).
        if REGISTER_REPULSION:
            _rep = model.pop_repulsion_loss()
            loss = loss + _rep
            accum_rep += float(_rep.detach()) / GRAD_ACCUM
        (loss / GRAD_ACCUM).backward()
        accum_ntp  += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg += float(v_reg.detach()) / GRAD_ACCUM

    if GRAD_CENTRALIZATION:
        for p in model.parameters():
            if p.grad is not None and p.grad.dim() >= 2:
                p.grad.sub_(p.grad.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True))

    if PER_GROUP_CLIP:
        grad_norm, _last_pg_norms = clip_grads_per_group(model)
    else:
        _last_pg_norms = per_group_grad_norms(model) if GRAD_SPIKE_DEBUG else {}
        if model.V_phi is not None:
            nn.utils.clip_grad_norm_(model.V_phi.parameters(), GRAD_CLIP_VPHI)
        grad_norm = nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
        )

    # ── per-step gradient-spike debugger ──
    if GRAD_SPIKE_DEBUG:
        _tot_preclip = float(grad_norm)
        if (_tot_preclip > GRAD_SPIKE_THRESHOLD
                and (step - _last_spike_step) >= GRAD_SPIKE_COOLDOWN):
            _last_spike_step = step
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(),
                              key=lambda kv: kv[1], reverse=True)[:8]
                _brk = '  '.join(f'{k}={v:.1f}' for k, v in _top)
            else:
                _brk = '(enable PER_GROUP_CLIP or GRAD_SPIKE_DEBUG for breakdown)'
            print(f'\n[spike] step {step+1}: pre-clip total grad={_tot_preclip:.1f}  '
                  f'ntp={accum_ntp:.3f}  v_reg={accum_vreg:.4f}')
            print(f'[spike]   top groups: {_brk}')
            _log_write(json.dumps({
                'step': step + 1, 'event': 'grad_spike',
                'pre_clip_grad_norm': round(_tot_preclip, 2),
                'ntp': round(accum_ntp, 4), 'v_reg': round(accum_vreg, 4),
                'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
            }) + '\n')

    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
        # No structured well bank in this arm, so there are no precision /
        # weight parameters to clamp back into range after the step.
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)

    # ── Watchdog ──
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh += 1
    else:
        _grad_norm_above_thresh = 0

    if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh} steps at step {step+1}.')
        if _last_pg_norms:
            _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
            print('[watchdog] top group norms (pre-clip): '
                  + ', '.join(f'{k}={v:.1f}' for k, v in _top))
        _log_write(json.dumps({
            'step': step + 1, 'event': 'watchdog_reload',
            'ema_grad_norm': round(_grad_norm_ema, 2),
            'above_thresh_steps': _grad_norm_above_thresh,
            'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
        }) + '\n')
        _reload_best()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh = 0
        n_skipped += 1

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        run_ntp, run_vreg, n_run = 0.0, 0.0, 0
        elapsed = time.time() - t0
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        _top_grp = ''
        if PER_GROUP_CLIP and _last_pg_norms:
            _k, _v = max(_last_pg_norms.items(), key=lambda kv: kv[1])
            _top_grp = f'top[{_k}]={_v:.1f}  '
        _vt_grp = ''
        if _last_pg_norms and 'V_theta' in _last_pg_norms:
            _vt_grp = f'vtheta={_last_pg_norms["V_theta"]:.2f}  '
        _rep_str = f'rep={accum_rep:.4f}  ' if REGISTER_REPULSION else ''
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  lr={lr_now:.2e}  '
            f'grad={float(grad_norm):.2f}  {_rep_str}{_vt_grp}{_top_grp}'
            f'gamma={model.gamma.item():.3f}  '
            f'alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)'
        )
        _log_write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'grad_norm_vtheta': _last_pg_norms.get('V_theta'),
            'gamma': model.gamma.item(), 'xi_alphas': alphas,
            'reg_repulsion': accum_rep,
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
              f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
              f'{marker}  ({elapsed:.0f}s)')
        _log_write(json.dumps({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }) + '\n')
        if is_best:
            save_checkpoint(step + 1, val_loss, tag_suffix='_best')

    if (step + 1) in set(CKPT_STEPS):
        if (step + 1) % EVAL_INTERVAL != 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
        save_checkpoint(step + 1, val_loss)

    if CAUSAL_PROBE_INTERVAL > 0 and (step + 1) % CAUSAL_PROBE_INTERVAL == 0:
        _cp_passed, _cp_delta = run_causal_probe(step + 1)
        _log_write(json.dumps({
            'step': step + 1,
            'causal_probe_passed': _cp_passed,
            'causal_probe_max_delta': _cp_delta,
        }) + '\n')

    if TRAINED_LEAK_PROBE_INTERVAL > 0 and (step + 1) % TRAINED_LEAK_PROBE_INTERVAL == 0:
        _tlp_result = run_trained_leak_probe(step + 1)
        _log_write(json.dumps(_tlp_result) + '\n')

_log_fh[0].close()
print(f'\nTraining complete. Best PPL: {best_val_ppl:.2f}')

## Fock v2.1 component diagnostics

Standalone probe — safe to run any time against the live model or a freshly
loaded checkpoint. It answers two questions:

1. **Structural health** — is each Fock piece being *used well*?
   (register diversity, utilisation, creation/reverse attention entropy,
   gate scale, destruction rate)
2. **PPL attribution** — how much does each piece actually *buy*?
   A runtime ablation zeros the reverse channel / disables the registers and
   measures the val-loss delta on a fixed set of batches.

The best improvement target is a component with **~0 ΔPPL AND an unhealthy
flag** (used, but not pulling its weight).

For this arm there is a third question worth reading off the same output: with
no Gaussian wells to supply structure, does the Fock register machinery pick up
more of the load? Compare `dPPL(- registers)` here against the Gaussian d768
run's value, and against the d384 MLP arm to see how the split moves with
scale.

In [ ]:
import torch
_bp = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
_bd = torch.load(_bp, map_location=DEVICE, weights_only=False)
model.load_state_dict(_bd['model_state_dict'], strict=False)
model.eval()
print(f"Probe target -> {_bp.name}  step {_bd.get('step')}  PPL {_bd.get('val_ppl'):.2f}")
del _bd
import gc; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()


# ── Fock v2.1 COMPONENT DIAGNOSTICS ──────────────────────────────
# (see markdown above). Reuses model / val_ids / get_batch / DEVICE.
import gc, math, sys, torch, numpy as np
# --- drop pinned tracebacks + leftover tensors from earlier OOMs ---
for _a in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _a): setattr(sys, _a, None)
for _n in ['_', '__', '___', '_batches', '_pb', 'rep', '_x0', '_y0', '_i', '_ii', '_iii']:
    if _n in globals(): globals()[_n] = None
model.zero_grad(set_to_none=True)
try: optim.zero_grad(set_to_none=True)
except Exception: pass
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU free {_free/1e9:.1f} / {_total/1e9:.1f} GB before probe')
PROBE_BS = 2                      # small batch: the (B,M,T,d) readout tensors dominate
_rng = np.random.default_rng(1234)
def _mk(n, bs):
    return [(torch.from_numpy(a).to(DEVICE), torch.from_numpy(b).to(DEVICE))
            for a, b in (get_batch(val_ids, bs, BLOCK_SIZE, _rng) for _ in range(n))]
def _eval_on(batches):
    model.eval(); losses = []
    for x, y in batches:
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(float(loss.item()))
        del loss
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return float(np.mean(losses))
# --- 1. structural health (one small captured forward) ---
model.eval(); model.set_fock_capture(True)
_hx, _hy = _mk(1, PROBE_BS)[0]
with torch.enable_grad():
    _out = model(_hx, _hy)
del _out                          # <-- don't pin the graph
rep = model.fock_component_report()
_cols = ['layer','active_frac','reg_cos_sim','create_entropy','create_alpha_max',
         'rev_entropy','rev_scale','qforce_ratio','destroy_mean']
print('='*72); print('Fock v2.1 STRUCTURAL HEALTH'); print('='*72)
print('  '.join(f'{c[:10]:>10}' for c in _cols))
for d in rep['per_layer']:
    print('  '.join(f'{str(d.get(c)):>10}' if isinstance(d.get(c),(bool,type(None)))
                    else f'{float(d.get(c)):>10.3f}' for c in _cols))
print('-'*72); print('summary:', {k: round(v,3) for k,v in rep['summary'].items()})
for f in rep.get('flags', []): print('  * '+f)
model.set_fock_capture(False)
del _hx, _hy, rep; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()
# --- 2. PPL attribution ---
_pb = _mk(40, PROBE_BS)
base = _eval_on(_pb); base_ppl = math.exp(base); rows = [('full model', base)]
if getattr(model, 'reverse_channel_scale', None) is not None:
    _s = model.reverse_channel_scale.detach().clone()
    with torch.no_grad(): model.reverse_channel_scale.zero_()
    rows.append(('  - reverse channel', _eval_on(_pb)))
    with torch.no_grad(): model.reverse_channel_scale.copy_(_s)
_thr = model.cfg.register_salience_threshold
try:
    model.cfg.register_salience_threshold = 1e9
    rows.append(('  - registers (all)', _eval_on(_pb)))
finally:
    model.cfg.register_salience_threshold = _thr
print('\n'+'='*72); print(f"{'arm':<22}{'loss':>10}{'ppl':>10}{'dPPL':>10}")
for n, l in rows:
    p = math.exp(l); print(f'{n:<22}{l:>10.4f}{p:>10.2f}{p-base_ppl:>+10.2f}')

## Run summary — loss curve and causal-leak history

Reads `training_log.jsonl` (append-only across every resume, so this covers the
whole run rather than just the current session) and renders:

- train NTP loss and val PPL against step,
- the `V_theta` per-group gradient norm, which is the direct evidence for or
  against the "MLP forces are smoother than Gaussian forces" claim,
- the history of both causal probes.

Safe to run mid-training from a second session.

In [ ]:
# ── Run summary ───────────────────────────────────────────────────
import json, math
import numpy as np
import matplotlib.pyplot as plt

_log = RESULTS_DIR / 'training_log.jsonl'
_train, _val, _vt_grad, _spikes = [], [], [], []
_arch_probes, _leak_probes = [], []

if not _log.exists():
    print(f'No log yet at {_log}')
else:
    with _log.open() as _f:
        for _line in _f:
            try:
                r = json.loads(_line)
            except json.JSONDecodeError:
                continue
            if r.get('event') == 'grad_spike':
                _spikes.append(r)
            elif 'val_ppl' in r:
                _val.append((r['step'], r['val_ppl']))
            elif 'train_loss' in r:
                _train.append((r['step'], r['train_loss']))
                if r.get('grad_norm_vtheta') is not None:
                    _vt_grad.append((r['step'], r['grad_norm_vtheta']))
            if 'causal_probe_passed' in r:
                _arch_probes.append(r)
            if 'ppl_last_pos_leak_free' in r:
                _leak_probes.append(r)

    # A resume re-logs its start step, so keep the last record per step.
    def _dedup(pairs):
        d = {}
        for s, v in pairs:
            d[s] = v
        return sorted(d.items())

    _train, _val, _vt_grad = _dedup(_train), _dedup(_val), _dedup(_vt_grad)

    print(f'Log: {_log}')
    print(f'  {len(_train):,} train points, {len(_val):,} evals, '
          f'{len(_spikes):,} grad spikes')
    if _val:
        _best_s, _best_p = min(_val, key=lambda sv: sv[1])
        print(f'  best val PPL {_best_p:.2f} @ step {_best_s:,}  '
              f'(latest: {_val[-1][1]:.2f} @ {_val[-1][0]:,})')

    _n_panels = 3 if _vt_grad else 2
    fig, axes = plt.subplots(1, _n_panels, figsize=(6 * _n_panels, 4.5))

    ax = axes[0]
    if _train:
        ax.plot([s for s, _ in _train], [v for _, v in _train],
                lw=0.8, alpha=0.7, label='train NTP')
    ax.set_xlabel('step'); ax.set_ylabel('loss (nats)')
    ax.set_title('Training loss — MLP V_theta d768 OWT')
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

    ax = axes[1]
    if _val:
        ax.plot([s for s, _ in _val], [v for _, v in _val],
                lw=1.2, color='crimson', label='val PPL')
    for _lp in _leak_probes:
        ax.scatter(_lp['step'], _lp['ppl_last_pos_leak_free'], marker='*',
                   s=90, color='green', zorder=5)
    if _leak_probes:
        ax.scatter([], [], marker='*', s=90, color='green',
                   label='honest PPL (leak-free)')
    ax.set_xlabel('step'); ax.set_ylabel('PPL')
    ax.set_title('Validation PPL')
    ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend(fontsize=8)

    if _vt_grad:
        ax = axes[2]
        ax.plot([s for s, _ in _vt_grad], [v for _, v in _vt_grad],
                lw=0.8, color='darkorange')
        ax.axhline(GRAD_CLIP, ls='--', color='grey', lw=1,
                   label=f'clip={GRAD_CLIP}')
        ax.set_xlabel('step'); ax.set_ylabel('pre-clip grad norm')
        ax.set_title('V_theta group gradient norm')
        ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend(fontsize=8)

    plt.tight_layout()
    _png = RESULTS_DIR / f'{CKPT_PREFIX}_run_summary.png'
    fig.savefig(_png, dpi=150)
    print(f'  Saved {_png}')
    plt.show()

    # ── Causal-leak audit history ──
    print('\n' + '=' * 78)
    print('CAUSAL-LEAK AUDIT HISTORY')
    print('=' * 78)
    if _arch_probes:
        print('\nStage 1 — architectural probe (max|dlogit| must be exactly 0.0):')
        print(f'  {"step":>9}  {"max|dlogit|":>13}  {"verdict":>8}')
        for r in _arch_probes:
            print(f'  {r["step"]:>9,}  {r["causal_probe_max_delta"]:>13.3e}  '
                  f'{"PASS" if r["causal_probe_passed"] else "FAIL":>8}')
    else:
        print('\nStage 1 — no architectural probe records yet.')

    if _leak_probes:
        print('\nStage 2 — trained-scale probe + honest PPL:')
        print(f'  {"step":>9}  {"standard PPL":>13}  {"honest PPL":>11}  '
              f'{"diff (nats)":>12}  {"max|dlogit|":>12}  {"verdict":>8}')
        for r in _leak_probes:
            _v = 'CLEAN' if r['paired_diff_nats'] < 0.1 else 'LEAK'
            print(f'  {r["step"]:>9,}  {r["ppl_mid_window_standard"]:>13.2f}  '
                  f'{r["ppl_last_pos_leak_free"]:>11.2f}  '
                  f'{r["paired_diff_nats"]:>+12.4f}  '
                  f'{r["probe_max_dlogit_past"]:>12.3e}  {_v:>8}')
        print('\n  "honest PPL" scores each target from the tokens strictly before it;')
        print('  "standard PPL" scores it mid-window, the way the training loss and')
        print('  the in-loop eval do. A gap is the perplexity the leak was worth.')
    else:
        print('\nStage 2 — no trained-scale probe records yet.')

    if _spikes:
        print(f'\nGradient spikes (pre-clip total > {GRAD_SPIKE_THRESHOLD}): {len(_spikes)}')
        for r in _spikes[-5:]:
            _tg = ', '.join(f'{k}={v}' for k, v in
                            list(r.get('top_groups', {}).items())[:3])
            print(f'  step {r["step"]:>8,}  norm={r["pre_clip_grad_norm"]:>9.1f}  {_tg}')
    else:
        print(f'\nNo gradient spikes above {GRAD_SPIKE_THRESHOLD} — '
              f'consistent with the bounded-Lipschitz argument for the MLP potential.')